Cell 2 — imports

In [6]:
import pandas as pd
import numpy as np
import joblib

from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

Cell 3 — set project paths correctly


In [8]:
from pathlib import Path

BASE_DIR = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd()

DATA_PATH = BASE_DIR / "data" / "nlp" / "sentiment140" / "training.csv"
MODEL_DIR = BASE_DIR / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print("BASE_DIR =", BASE_DIR)
print("DATA_PATH =", DATA_PATH)
print("DATA_PATH exists? =", DATA_PATH.exists())
print("MODEL_DIR =", MODEL_DIR)

BASE_DIR = D:\OneDrive - FORSAN FOODS & CONSUMER PRODUCTS\AI Sandiego\Capstone_Project\fitness-journey-analyzer
DATA_PATH = D:\OneDrive - FORSAN FOODS & CONSUMER PRODUCTS\AI Sandiego\Capstone_Project\fitness-journey-analyzer\data\nlp\sentiment140\training.csv
DATA_PATH exists? = True
MODEL_DIR = D:\OneDrive - FORSAN FOODS & CONSUMER PRODUCTS\AI Sandiego\Capstone_Project\fitness-journey-analyzer\models


Cell 4


In [9]:
col_names = ["target", "id", "date", "flag", "user", "text"]

df = pd.read_csv(
    DATA_PATH,
    encoding="latin-1",
    header=None,
    names=col_names
)

print("Shape:", df.shape)
df.head()

Shape: (1600000, 6)


,target,id,date,flag,user,text
0,0,1467810369,Mon Apr 06 22:19:45 PDT 2009,NO_QUERY,_TheSpecialOne_,"@switchfoot http://twitpic.com/2y1zl - Awww, t..."
1,0,1467810672,Mon Apr 06 22:19:49 PDT 2009,NO_QUERY,scotthamilton,is upset that he can't update his Facebook by ...
2,0,1467810917,Mon Apr 06 22:19:53 PDT 2009,NO_QUERY,mattycus,@Kenichan I dived many times for the ball. Man...
3,0,1467811184,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,ElleCTF,my whole body feels itchy and like its on fire
4,0,1467811193,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,Karoli,"@nationwideclass no, it's not behaving at all...."


Cell 5

In [10]:
df = df[["target", "text"]].copy()

print(df.shape)
df.head()

(1600000, 2)


,target,text
0,0,"@switchfoot http://twitpic.com/2y1zl - Awww, t..."
1,0,is upset that he can't update his Facebook by ...
2,0,@Kenichan I dived many times for the ball. Man...
3,0,my whole body feels itchy and like its on fire
4,0,"@nationwideclass no, it's not behaving at all...."


Cell 6

In [11]:
df_pos = df[df["target"] == 4].sample(20000, random_state=42)
df_neg = df[df["target"] == 0].sample(20000, random_state=42)

df_sample = pd.concat([df_pos, df_neg], axis=0).sample(frac=1, random_state=42).reset_index(drop=True)

print("Sample shape:", df_sample.shape)
print(df_sample["target"].value_counts())
df_sample.head()

Sample shape: (40000, 2)
target
0    20000
4    20000
Name: count, dtype: int64


,target,text
0,0,BOO! Randi Zuckerburg didn't make it today St...
1,4,@tova_s I see you've regained your twitter acc...
2,0,@mumphlett - What's up? I am here..Have a lil ...
3,4,@jtweeti Hi there! I just realised that it's b...
4,0,@skybreaker Ah Im sorry that Photoshop is no g...


Cell 7

In [12]:
df_sample["sentiment_label"] = df_sample["target"].map({0: "negative", 4: "positive"})

df_sample = df_sample.dropna(subset=["text", "sentiment_label"]).copy()
df_sample["text"] = df_sample["text"].astype(str).str.strip()
df_sample = df_sample[df_sample["text"] != ""]

print(df_sample.shape)
print(df_sample["sentiment_label"].value_counts())
df_sample.head()

(40000, 3)
sentiment_label
negative    20000
positive    20000
Name: count, dtype: int64


,target,text,sentiment_label
0,0,BOO! Randi Zuckerburg didn't make it today St...,negative
1,4,@tova_s I see you've regained your twitter acc...,positive
2,0,@mumphlett - What's up? I am here..Have a lil ...,negative
3,4,@jtweeti Hi there! I just realised that it's b...,positive
4,0,@skybreaker Ah Im sorry that Photoshop is no g...,negative


Cell 8

In [13]:
X = df_sample["text"]
y = df_sample["sentiment_label"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train size:", len(X_train))
print("Test size:", len(X_test))

Train size: 32000
Test size: 8000


Cell 9

In [14]:
sentiment_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        stop_words="english",
        ngram_range=(1, 2),
        max_features=10000
    )),
    ("clf", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

sentiment_pipeline.fit(X_train, y_train)
print("Model training completed.")

Model training completed.


Cell 10

In [17]:
y_pred = sentiment_pipeline.predict(X_test)

print("Sentiment Accuracy:", accuracy_score(y_test, y_pred))
print()
print(classification_report(y_test, y_pred))

Sentiment Accuracy: 0.743625

              precision    recall  f1-score   support

    negative       0.75      0.73      0.74      4000
    positive       0.74      0.76      0.75      4000

    accuracy                           0.74      8000
   macro avg       0.74      0.74      0.74      8000
weighted avg       0.74      0.74      0.74      8000



Cell 11

In [18]:
sample_texts = [
    "I feel strong and energetic after workout",
    "I am exhausted and my legs are sore",
    "Great cardio session today",
    "Terrible workout, I feel drained",
    "I feel amazing after today's gym session"
]

preds = sentiment_pipeline.predict(sample_texts)
probs = sentiment_pipeline.predict_proba(sample_texts)

for text, pred, prob in zip(sample_texts, preds, probs):
    print("TEXT:", text)
    print("PREDICTION:", pred)
    print("CONFIDENCE:", round(max(prob), 4))
    print("-" * 60)

TEXT: I feel strong and energetic after workout
PREDICTION: negative
CONFIDENCE: 0.523
------------------------------------------------------------
TEXT: I am exhausted and my legs are sore
PREDICTION: negative
CONFIDENCE: 0.9324
------------------------------------------------------------
TEXT: Great cardio session today
PREDICTION: positive
CONFIDENCE: 0.8692
------------------------------------------------------------
TEXT: Terrible workout, I feel drained
PREDICTION: negative
CONFIDENCE: 0.7633
------------------------------------------------------------
TEXT: I feel amazing after today's gym session
PREDICTION: positive
CONFIDENCE: 0.7046
------------------------------------------------------------


Cell 12

In [19]:
MODEL_PATH = MODEL_DIR / "sentiment_model.pkl"

joblib.dump(sentiment_pipeline, MODEL_PATH)

print("Saved model to:")
print(MODEL_PATH)
print("Exists?", MODEL_PATH.exists())

Saved model to:
D:\OneDrive - FORSAN FOODS & CONSUMER PRODUCTS\AI Sandiego\Capstone_Project\fitness-journey-analyzer\models\sentiment_model.pkl
Exists? True


Cell 13

In [20]:
loaded_model = joblib.load(MODEL_DIR / "sentiment_model.pkl")

test_text = ["I feel very good after my workout"]
pred = loaded_model.predict(test_text)[0]
prob = max(loaded_model.predict_proba(test_text)[0])

print("Prediction:", pred)
print("Confidence:", round(prob, 4))

Prediction: negative
Confidence: 0.5886
